In [3]:
import os
os.environ['KAGGLE_USERNAME'] = "danish111122345"
os.environ['KAGGLE_KEY'] = "0260bfd3a64ca29bcb79affab8d43512"

In [4]:
!kaggle datasets download -d 'aliiihussain/amazon-sales-dataset'

Dataset URL: https://www.kaggle.com/datasets/aliiihussain/amazon-sales-dataset
License(s): CC0-1.0
100% 1.24M/1.24M [00:00<00:00, 139MB/s]



In [8]:
import pandas as pd
df = pd.read_csv('amazon-sales-dataset.zip')

In [ ]:
df.shape

(50000, 13)

In [ ]:
df.isnull().sum()

,0
order_id,0
order_date,0
product_id,0
product_category,0
price,0
discount_percent,0
quantity_sold,0
customer_region,0
payment_method,0
rating,0


In [17]:
df['order_date'] = pd.to_datetime(df['order_date'])

df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day
df['order_day_of_week'] = df['order_date'].dt.dayofweek
df['order_week_of_year'] = df['order_date'].dt.isocalendar().week.astype(int)

display(df.head())

,order_id,order_date,product_id,product_category,price,discount_percent,quantity_sold,customer_region,payment_method,rating,review_count,discounted_price,total_revenue,order_year,order_month,order_day,order_day_of_week,order_week_of_year,hist_avg_qty_category,hist_avg_price_category
11227,11228,2022-01-01,2772,Fashion,203.51,0,3,Asia,UPI,2.3,220,203.51,610.53,2022,1,1,5,52,2.9994,218.886566
14120,14121,2022-01-01,1814,Books,481.13,5,3,Middle East,Wallet,2.5,5,457.07,1371.21,2022,1,1,5,52,2.9994,218.886566
36850,36851,2022-01-01,4847,Fashion,163.38,5,4,Asia,UPI,3.4,78,155.21,620.84,2022,1,1,5,52,3.0000,203.510000
26743,26744,2022-01-01,1370,Home & Kitchen,248.48,5,2,Middle East,Wallet,2.9,43,236.06,472.12,2022,1,1,5,52,2.9994,218.886566
7676,7677,2022-01-01,4542,Fashion,10.47,5,3,Asia,UPI,3.6,398,9.95,29.85,2022,1,1,5,52,3.5000,179.360000


In [22]:
df = df.sort_values(by='order_date')

df['hist_avg_qty_category'] = df.groupby('product_category')['quantity_sold'].transform(lambda x: x.expanding().mean().shift(1))

df['hist_avg_price_category'] = df.groupby('product_category')['discounted_price'].transform(lambda x: x.expanding().mean().shift(1))

df['hist_avg_qty_category'] = df['hist_avg_qty_category'].fillna(df['quantity_sold'].mean())
df['hist_avg_price_category'] = df['hist_avg_price_category'].fillna(df['discounted_price'].mean())

display(df.head())

,order_id,order_date,product_id,product_category,price,discount_percent,quantity_sold,customer_region,payment_method,rating,review_count,discounted_price,total_revenue,order_year,order_month,order_day,order_day_of_week,order_week_of_year,hist_avg_qty_category,hist_avg_price_category
11227,11228,2022-01-01,2772,Fashion,203.51,0,3,Asia,UPI,2.3,220,203.51,610.53,2022,1,1,5,52,2.9994,218.886566
5169,5170,2022-01-01,4296,Home & Kitchen,226.97,15,5,Europe,Cash on Delivery,1.8,37,192.92,964.60,2022,1,1,5,52,2.9994,218.886566
2032,2033,2022-01-01,4802,Books,340.18,5,3,Europe,Credit Card,3.4,480,323.17,969.51,2022,1,1,5,52,2.9994,218.886566
24608,24609,2022-01-01,2933,Home & Kitchen,329.38,5,3,Middle East,Wallet,2.3,380,312.91,938.73,2022,1,1,5,52,5.0000,192.920000
20241,20242,2022-01-01,1194,Electronics,401.01,30,1,North America,UPI,4.6,497,280.71,280.71,2022,1,1,5,52,2.9994,218.886566


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

y = df['quantity_sold']

x_cols = ['product_category', 'price', 'discount_percent', 'customer_region', 'payment_method', 'rating',
          'review_count', 'discounted_price', 'order_year', 'order_month', 'order_day',
          'order_day_of_week', 'order_week_of_year', 'hist_avg_qty_category', 'hist_avg_price_category']
x = df[x_cols]

categorical_features = ['product_category', 'customer_region', 'payment_method']
numerical_features = [col for col in x_cols if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

x_processed = preprocessor.fit_transform(x)
x_train, x_test, y_train, y_test = train_test_split(x_processed, y, test_size=0.2, random_state=42)

print(f"Shape of processed features: {x_processed.shape}")
print(f"Shape of x_train: {x_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of x_test: {x_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of processed features: (50000, 27)
Shape of x_train: (40000, 27)
Shape of y_train: (40000,)
Shape of x_test: (10000, 27)
Shape of y_test: (10000,)


In [20]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(x_train, y_train)

y_pred = xgb_model.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
display(predictions_df.head(10))

Mean Absolute Error (MAE): 1.22
Root Mean Squared Error (RMSE): 1.42


,Actual,Predicted
35616,3,2.973250
28906,5,3.066721
29239,1,2.876545
631,2,3.205252
23958,1,3.080034
10300,4,2.981093
8483,2,2.883676
25736,1,2.912403
10403,1,3.022792
31526,2,3.010482


In [21]:
import pandas as pd

new_data = pd.DataFrame({
    'product_category': ['Books'],
    'price': [150.0],
    'discount_percent': [10],
    'customer_region': ['North America'],
    'payment_method': ['Credit Card'],
    'rating': [4.5],
    'review_count': [500],
    'discounted_price': [135.0],
    'order_year': [2023],
    'order_month': [11],
    'order_day': [15],
    'order_day_of_week': [2],
    'order_week_of_year': [46],
    'hist_avg_qty_category': [3.5],
    'hist_avg_price_category': [200.0]
})

new_data_processed = preprocessor.transform(new_data)

predicted_quantity_sold = xgb_model.predict(new_data_processed)

print(f"Predicted Quantity Sold for the given values: {predicted_quantity_sold[0]:.2f} units")

Predicted Quantity Sold for the given values: 2.46 units
